In [ ]:
import json
import os
import pprint

import dotenv
import requests

In [ ]:
dotenv.load_dotenv()
OPENSEARCH_HOST = "https://localhost:10004"
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")

### 登録する

##### agentを作成
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/index/#step-4-create-an-agent
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/agent-customization/#agent-interface-configuration
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/flow-agent/

In [ ]:
LLM_OPENSEARCH_MODEL_ID = "clb2tp8B6uqHd5u0ERW5"  # claude 4.6 sonnetで作成したML Model

In [ ]:
# https://docs.opensearch.org/latest/ml-commons-plugin/agents-tools/tools/query-planning-tool/#register-parameters
agent_payload = {
    "name": "underwear-search-agent",
    "type": "flow",
    "description": "Agent for underwear product search",
    "tools": [
        {
            "type": "QueryPlanningTool",
            "parameters": {
                "model_id": LLM_OPENSEARCH_MODEL_ID,
                "response_filter": "$.output.message.content[0].text",
                "query_planner_system_prompt": (
                    "Return ONLY a valid OpenSearch search request body.\n"
                    "Rules:\n"
                    " - Return exactly one JSON object.\n"
                    " - Do not use Markdown.\n"
                    " - Do not use code fences.\n"
                    " - Do not include explanations.\n"
                    " - The response must be valid JSON.\n"
                    " - Do not include '_source'.\n"
                    " - Always set 'size' to 5 unless another value is explicitly requested.\n\n"

                    "Index schema:\n"
                    " - product: keyword, not indexed. Do not search this field.\n"
                    " - category: keyword. Use for structured filtering such as ブラジャー or ショーツ.\n"
                    " - name: text. Use for BM25 lexical search.\n"
                    " - name_vector: knn_vector. Use for semantic search.\n"
                    " - description: text. Use for BM25 lexical search.\n"
                    " - description_vector: knn_vector. Use for semantic search.\n"
                    " - detail: text. Use for BM25 lexical search.\n"
                    " - detail_vector: knn_vector. Use for semantic search.\n"
                    " - image_url: keyword, not indexed. Do not search this field.\n"
                    " - image_vector: knn_vector. Use for semantic visual search.\n"
                    " - product_url: keyword, not indexed. Do not search this field.\n"
                    " - size_price: nested. Each element contains:\n"
                    "     - size: keyword\n"
                    "     - price: float\n\n"

                    "Important:\n"
                    " - There is NO color field and NO color_vector field in this index.\n"
                    " - Never generate queries against color or color_vector.\n"
                    " - Color-related search intent should primarily be handled by image_vector and, when useful, BM25 search against name, description, or detail.\n\n"

                    "Search strategy:\n"
                    " - Use a bool query.\n"
                    " - Do NOT use hybrid queries.\n"
                    " - Combine lexical and semantic search clauses under bool.should.\n"
                    " - Set minimum_should_match to 1 when bool.should is used.\n"
                    " - Use bool.filter only for strict structured constraints.\n"
                    " - Structured constraints include category, exact size, and price conditions.\n"
                    " - Do not use filter for vague semantic concepts such as elegant, cute, sexy, beautiful, comfortable, or fashionable.\n"
                    " - BM25 and semantic search should complement each other.\n\n"

                    "Category:\n"
                    " - This index contains both ブラジャー and ショーツ.\n"
                    " - If the user explicitly requests ブラジャー, filter category to ブラジャー.\n"
                    " - If the user explicitly requests ショーツ, filter category to ショーツ.\n"
                    " - If the user explicitly requests 下着, インナー, ランジェリー, or does not specify a product category, do not unnecessarily filter category.\n"
                    " - Never use category as a semantic search field.\n\n"

                    "Semantic search:\n"
                    " - When semantic understanding is beneficial, generate neural queries for:\n"
                    "   - image_vector\n"
                    "   - name_vector\n"
                    "   - description_vector\n"
                    "   - detail_vector\n"
                    " - Always include model_id, k, and boost.\n"
                    " - Recommended values:\n"
                    "   - image_vector: boost 3.0-5.0, k=15\n"
                    "   - name_vector: boost 1.0-2.0, k=10\n"
                    "   - description_vector: boost 1.0-2.0, k=10\n"
                    "   - detail_vector: boost 1.0-2.0, k=10\n"
                    " - Adjust boost dynamically according to the user's intent.\n\n"

                    "Image semantic search:\n"
                    " - Prioritize image_vector for visually observable characteristics.\n"
                    " - This includes color, color tone, brightness, darkness, vividness, pattern, lace appearance, floral appearance, design, silhouette, shape, visual impression, elegance, cuteness, sexiness, and overall atmosphere.\n"
                    " - For queries such as '赤系のブラジャー', '黒で大人っぽいブラジャー', '淡い色のショーツ', or '華やかなデザイン', prioritize image_vector.\n"
                    " - Do not attempt to convert color expressions into an exact keyword filter because there is no indexed color field.\n"
                    " - Image search should be considered especially important when the user's intent is primarily visual.\n\n"

                    "Text semantic search:\n"
                    " - Prioritize name_vector when the product name or product type is important.\n"
                    " - Prioritize description_vector for general semantic understanding of product descriptions and product concepts.\n"
                    " - Prioritize detail_vector for materials, functions, washing instructions, pads, wires, hooks, construction, and detailed product specifications.\n"
                    " - When the query contains multiple intents, combine appropriate neural queries with different boosts.\n\n"

                    "Lexical search:\n"
                    " - When the query contains important keywords, product names, product categories, materials, functions, attributes, or product specifications, generate BM25 queries using match or multi_match.\n"
                    " - Search name, description, and detail using BM25 when textual keyword matching is useful.\n"
                    " - Do not search product, image_url, or product_url.\n"
                    " - Do not search color or color_vector because these fields do not exist.\n"
                    " - Combine BM25 queries with neural queries under bool.should when appropriate.\n\n"

                    "Nested size and price search:\n"
                    " - size_price is a nested field.\n"
                    " - NEVER search size_price.size or size_price.price outside a nested query.\n"
                    " - When the user specifies a size, use a nested query with path 'size_price'.\n"
                    " - When the user specifies a price condition without a size, use a nested query on size_price.price.\n"
                    " - When the user specifies both size and price, BOTH conditions must be placed inside the SAME nested query so that they apply to the same size_price object.\n"
                    " - For example, 'C70で15000円以下' must match an object where size='C70' AND price<=15000.\n"
                    " - Do not combine size and price conditions across separate nested queries because that could match different size_price objects.\n"
                    " - Exact size requests should use term on size_price.size.\n"
                    " - Price conditions should use range on size_price.price.\n\n"

                    "Price handling:\n"
                    " - Price is not a top-level field.\n"
                    " - All price filtering must be performed through size_price.price inside a nested query.\n"
                    " - Never generate a top-level price filter.\n"
                    " - If the user specifies a size and price, ensure that the price condition applies to that requested size.\n"
                    " - Examples:\n"
                    "   - '15000円以下' -> price <= 15000\n"
                    "   - '15000円未満' -> price < 15000\n"
                    "   - '10000円以上' -> price >= 10000\n"
                    "   - '10000円〜20000円' -> price >= 10000 AND price <= 20000\n\n"

                    "Size handling:\n"
                    " - Sizes are stored as exact keyword values in size_price.size.\n"
                    " - For bra sizes such as B65, C70, D75, F80, use exact term matching.\n"
                    " - For panty sizes such as M, L, LL, XL, use exact term matching.\n"
                    " - If the user asks for a size, do not use semantic search to determine size availability. Use the nested size filter.\n"
                    " - If the user asks for both size and price, use the same nested query for both conditions.\n\n"

                    "Query examples:\n"
                    " - '赤系のブラジャー' -> filter category='ブラジャー', prioritize image_vector, and optionally use BM25 against name/description/detail.\n"
                    " - '黒で大人っぽいブラジャー' -> filter category='ブラジャー', prioritize image_vector and use semantic/BM25 search for the overall design intent.\n"
                    " - '華やかなショーツ' -> filter category='ショーツ' and prioritize image_vector.\n"
                    " - 'C70のブラジャー' -> filter category='ブラジャー' and use a nested size_price query with size='C70'.\n"
                    " - 'C70で15000円以下のブラジャー' -> filter category='ブラジャー' and use ONE nested query containing size='C70' AND price<=15000.\n"
                    " - 'Mサイズのショーツ' -> filter category='ショーツ' and use a nested size_price query with size='M'.\n"
                    " - 'レース素材のブラジャー' -> filter category='ブラジャー' and combine BM25/neural search against detail and description.\n"
                    " - '15000円以下の下着' -> use a nested price query; do not assume a top-level price field.\n"
                    " - '淡いピンクでかわいいブラジャー' -> filter category='ブラジャー' and strongly prioritize image_vector for visual appearance and color.\n\n"

                    "Final requirements:\n"
                    " - Generate only the OpenSearch search request body.\n"
                    " - Do not generate index names, URLs, Python code, or explanations.\n"
                    " - Do not use fields that are not present in the schema.\n"
                    " - Never use color or color_vector.\n"
                    " - Never use size_price fields outside a nested query.\n"
                    " - Preserve the user's search intent.\n"
                    " - Use structured filters for hard constraints and BM25/neural search for semantic preferences."
                )
            }
        }
    ]
}

In [ ]:
agent_register_url = "{a}/_plugins/_ml/agents/_register".format(a=OPENSEARCH_HOST)
response = requests.post(url=agent_register_url,
                         auth=(OPENSEARCH_USER,
                               OPENSEARCH_PASSWORD),
                         headers={"Content-Type": "application/json"},
                         json=agent_payload,
                         verify=False
                        )
print(response.status_code)
print(response.json())

In [ ]:
agent_id = response.json()["agent_id"]
# agent_id = "Jbh0V6ABeg8Lu8jYHeeV"

##### agentをテストする
##### https://docs.opensearch.org/latest/ml-commons-plugin/agents-tools/tools/query-planning-tool/#execute-parameters

In [ ]:
agent_test_url = "{a}/_plugins/_ml/agents/{b}/_execute".format(a=OPENSEARCH_HOST, b=agent_id)
# 03_opensearch_index_control.ipynbで作ったもの
target_index_name = "bra_panty_product_database"
# 01_opensearch_model_control.ipynbで作ったもの
embedding_ml_model_id = "7LIfsp8BXwdvdGgU_cbQ"

In [ ]:
agent_test_payload = {"parameters":{"question": "20,000円程度でオシャレなブラジャーとショーツを提案頂けますか？両者のデザインは一貫していると嬉しいです。",
                                    "index_name": target_index_name,
                                    "embedding_model_id": embedding_ml_model_id},
                      "dsl_query": True
                     }

In [ ]:
agent_test_response = requests.post(url=agent_test_url,
                                    auth=(OPENSEARCH_USER,
                                          OPENSEARCH_PASSWORD),
                                    headers={"Content-Type": "application/json"},
                                    json=agent_test_payload,
                                    verify=False
                                   )
print(agent_test_response.status_code)
pprint.pprint(agent_test_response.json())

##### pipelineを作成
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/neural-search/?utm_source=chatgpt.com#step-2c-create-a-search-pipeline
##### https://docs.opensearch.org/latest/search-plugins/search-pipelines/agentic-context-processor/#example

In [ ]:
EMBEDDING_OPENSEARCH_MODEL_ID = "7LIfsp8BXwdvdGgU_cbQ"  # amazon nova multimodal embeddingで作成したML Model

In [ ]:
pipeline_payload = {"description": "bra and panty product agentic search pipeline",
                    "request_processors": [{"agentic_query_translator": {"agent_id": agent_id,
                                                                         "embedding_model_id": EMBEDDING_OPENSEARCH_MODEL_ID}
                                           }],
                    "response_processors": [{"agentic_context": {"dsl_query": True,
                                                                 "agent_steps_summary": True}
                                            }]
                   }

In [ ]:
pipeline_name = "my-bra-and-panty-agentic-search-pipeline"
pipeline_register_url = "{a}/_search/pipeline/{b}".format(a=OPENSEARCH_HOST, b=pipeline_name)
response_2 = requests.put(url=pipeline_register_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          headers={"Content-Type": "application/json"},
                          json=pipeline_payload,
                          verify=False
                         )
print(response_2.status_code)
print(response_2.json())

##### 作成済みのagentやpipelineを確認する場合

In [ ]:
# https://docs.opensearch.org/latest/ml-commons-plugin/api/agent-apis/search-agent
agent_check_url = "{a}/_plugins/_ml/agents/_search".format(a=OPENSEARCH_HOST)
agent_check_payload = {"query": {"match_all": {}},
                       "size": 1000}
response_3 = requests.post(url=agent_check_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           headers={"Content-Type": "application/json"},
                           json=agent_check_payload,
                           verify=False)
print(response_3.status_code)
pprint.pprint(response_3.json())

In [ ]:
# https://docs.opensearch.org/latest/search-plugins/search-pipelines/retrieving-search-pipeline/
pipeline_check_url = "{a}/_search/pipeline".format(a=OPENSEARCH_HOST)
response_4 = requests.get(url=pipeline_check_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          headers={"Content-Type": "application/json"},
                          verify=False)
print(response_4.status_code)
pprint.pprint(response_4.json())

### 削除する

##### pipelineを削除
##### https://docs.opensearch.org/latest/search-plugins/search-pipelines/deleting-search-pipeline/

In [ ]:
delete_pipeline_name = "my-bra-and-panty-agentic-search-pipeline"

In [ ]:
pipeline_delete_url = "{a}/_search/pipeline/{b}".format(a=OPENSEARCH_HOST, b=delete_pipeline_name)
response_5 = requests.delete(pipeline_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_5.status_code)
print(response_5.json())

##### agentを削除
##### https://docs.opensearch.org/latest/ml-commons-plugin/api/agent-apis/delete-agent/

In [ ]:
delete_agent_id = "Jbh0V6ABeg8Lu8jYHeeV"

In [ ]:
agent_delete_url = "{a}/_plugins/_ml/agents/{b}".format(a=OPENSEARCH_HOST, b=delete_agent_id)
response_6 = requests.delete(agent_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_6.status_code)
print(response_6.json())